# Small-circuit scaling sweep (finite data, multi-epoch)

One output (wire 9, tap depth 4, depends on all 10 inputs) of a 10-wire depth-4 circuit. Inputs are split 80/20 into a train pool and a held-out set. Training passes through the pool in a freshly shuffled order each epoch, and held-out loss is evaluated once per epoch.

With a constant LR, one `EPOCHS`-epoch run gives every epoch's checkpoint for free. So the LR sweep runs are the scaling runs too: each plot takes the best LR per shape *at that epoch*. Runs are idempotent.

In [ ]:
import os
import sys

if "google.colab" in sys.modules:
    %pip -q install -U "jax[cuda12]" optax
    if not os.path.exists("/content/circscale"):
        !git clone https://github.com/amdson/circscale.git /content/circscale
    %cd /content/circscale
    !git pull
    from google.colab import drive
    drive.mount("/content/drive")
    os.makedirs("/content/drive/MyDrive/circscale_runs", exist_ok=True)
    if not os.path.islink("runs"):
        os.symlink("/content/drive/MyDrive/circscale_runs", "runs")

import jax
print(jax.devices())

In [ ]:
import numpy as np

from train import RunConfig, run

N_WIRES, CIRC_DEPTH, CIRCUIT_SEED, TARGET_WIRE = 10, 4, 0, 9
TRAIN_FRAC, BATCH = 0.8, 32
EPOCHS = 5                                                     # trained per run
EPOCH_STEPS = int(round(TRAIN_FRAC * 2 ** N_WIRES)) // BATCH   # 25
GRID = [(32, 2), (48, 2), (64, 3), (96, 3), (128, 4),
        (180, 5), (256, 6), (360, 7), (512, 8)]
LR_GRID = [1e-4, 3e-4, 1e-3, 3e-3, 1e-2]
OUT_DIR = f"runs/small_c{N_WIRES}x{CIRC_DEPTH}"


def cfg(width, depth, lr, seed=0):
    return RunConfig(width=width, mlp_depth=depth, lr=lr, warmup=0, batch=BATCH,
                     steps=EPOCHS * EPOCH_STEPS, eval_every=EPOCH_STEPS,
                     n_wires=N_WIRES, circ_depth=CIRC_DEPTH, circuit_seed=CIRCUIT_SEED,
                     output_wires=(TARGET_WIRE,), train_frac=TRAIN_FRAC,
                     data_order="epoch", model_seed=seed, out_dir=OUT_DIR)

## LR sweep (these are also the scaling runs)

In [ ]:
for w, d in GRID:
    for lr in LR_GRID:
        run(cfg(w, d, lr))

## Results

In [ ]:
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

from train import load_run


def n_params(w, d):
    return d * (w + 8 * w * w) + w  # non-embedding params


def fit_plot(ax, N, L):
    """Scatter L vs N with a fitted L = E + A N^-alpha."""
    N, L = np.array(N), np.array(L)
    ax.plot(N, L, "o")
    f = lambda lN, lE, lA, a: np.log(np.exp(lE) + np.exp(lA - a * lN))
    try:
        (lE, lA, a), _ = curve_fit(f, np.log(N), np.log(L),
                                   p0=(np.log(0.9 * L.min()), 1.0, 0.3), maxfev=20000)
        xs = np.logspace(np.log10(N.min()), np.log10(N.max()), 100)
        ax.plot(xs, np.exp(lE) + np.exp(lA) * xs ** -a, "k--",
                label=f"L = {np.exp(lE):.3f} + {np.exp(lA):.3g} N^-{a:.2f}")
        ax.legend(fontsize=8)
    except (RuntimeError, ValueError, TypeError):
        print("fit failed (too few points?)")
    ax.set(xscale="log", yscale="log", xlabel="non-embedding params N")


def ho_loss(w, d, lr, epoch):
    """Held-out target loss after `epoch` epochs."""
    _, r = load_run(cfg(w, d, lr).npz_path)
    i = np.flatnonzero(r["eval_steps"] == epoch * EPOCH_STEPS)[0]
    return float(r["per_out_loss_ho"][i, TARGET_WIRE])


def plot_at_epoch(epoch):
    N, L = [], []
    for w, d in GRID:
        losses = {lr: ho_loss(w, d, lr, epoch) for lr in LR_GRID
                  if cfg(w, d, lr).npz_path.exists()}
        if losses:
            best = min(losses, key=losses.get)
            print(f"w{w}d{d}: best lr {best:g}, held-out loss {losses[best]:.4f}")
            N.append(n_params(w, d)); L.append(losses[best])
    fig, ax = plt.subplots(figsize=(6, 4.2))
    fit_plot(ax, N, L)
    ax.axhline(np.log(2), color="gray", ls=":", lw=0.8)
    ax.set(ylabel="held-out BCE (best LR per shape)", title=f"after {epoch} epoch(s)")

In [ ]:
plot_at_epoch(1)

In [ ]:
K = 5   # <= EPOCHS
plot_at_epoch(K)